# Task 4: Sentiment Analysis

In this notebook, we address **Task 4** by analyzing text data to classify it as positive, negative, or neutral using Natural Language Processing (NLP).

We will use the **VADER** lexicon-based approach, which is especially robust for social media text and reviews.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

analyzer = SentimentIntensityAnalyzer()

## 1. Load the Target Text Data
We will attempt to load the custom scraped news dataset first. If it's not available, we'll fall back to YouTube video descriptions.

In [ ]:
file_path_news = '../data/ev_news_scraped.csv'
file_path_yt = '../data/youtube_videos.csv'

text_column = ''
if os.path.exists(file_path_news):
    print("Loading Scraped EV News...")
    df = pd.read_csv(file_path_news)
    text_column = 'title' # We analyze the titles of the news articles
else:
    print("Loading YouTube Data...")
    df = pd.read_csv(file_path_yt)
    text_column = 'title' # Analyze video titles

# Ensure data is string
df[text_column] = df[text_column].astype(str)

print(f"Analyzing {len(df)} records based on column: '{text_column}'")

## 2. Apply Sentiment Classification
We calculate the `compound` score using VADER. 
- **Positive**: compound >= 0.05
- **Negative**: compound <= -0.05
- **Neutral**: -0.05 < compound < 0.05

In [ ]:
def classify_sentiment(text):
    if not text.strip():
        return 'Neutral'
        
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_category'] = df[text_column].apply(classify_sentiment)
df['sentiment_score'] = df[text_column].apply(lambda x: analyzer.polarity_scores(x)['compound'])

# Display some examples
display(df[[text_column, 'sentiment_category', 'sentiment_score']].head())

## 3. Visualize Sentiment Distribution
What is the overall public opinion or tone of the text gathered?

In [ ]:
sentiment_counts = df['sentiment_category'].value_counts()

plt.figure(figsize=(8, 6))
colors = ['#2ca02c', '#1f77b4', '#d62728'] # Green (Pos), Blue (Neu), Red (Neg)

plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', 
        colors=[colors[['Positive', 'Neutral', 'Negative'].index(cat)] for cat in sentiment_counts.index],
        startangle=140, explode=[0.05]*len(sentiment_counts))

plt.title('Overall Sentiment Distribution', fontsize=15)
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.tight_layout()
plt.show()

## 4. Extracting Business Insights
By looking at the distribution, we can quickly inform marketing or product teams of the general attitude towards the Electric Vehicle topics tracked.

In [ ]:
print("Top 3 Most Positive Texts:")
for idx, row in df.sort_values(by='sentiment_score', ascending=False).head(3).iterrows():
    print(f"- [{row['sentiment_score']}] {row[text_column]}")

print("\nTop 3 Most Negative Texts:")
for idx, row in df.sort_values(by='sentiment_score', ascending=True).head(3).iterrows():
    print(f"- [{row['sentiment_score']}] {row[text_column]}")